# Estimación Bayesiana — Exploración interactiva
## Tópicos de Estadística Avanzada · UNS
**Dra. Beatriz Marrón — Dr. José M. Bavio**

Este notebook permite explorar cómo la elección de la distribución **a priori** 
afecta la distribución **a posteriori** y los **estimadores bayesianos**, 
comparándolos con los estimadores clásicos (máxima verosimilitud).

### Estructura
1. Librerías y configuración
2. **Ejemplo 1** — Bernoulli–Beta (prior conjugada)
3. **Ejemplo 2** — Exponencial–Gamma (prior conjugada)
4. **Ejemplo 3** — Normal–Normal (prior conjugada)

### Cómo usar
- Ejecutá todas las celdas (**Runtime → Run all** en Colab).
- Usá los **sliders** para modificar los hiperparámetros de la prior y el tamaño muestral.
- Observá cómo la posterior se actualiza y cómo el estimador bayesiano se acerca 
  al clásico a medida que **n crece** (dominio de los datos sobre la prior).

> 📖 Referencia: Cap. 10 de los apuntes de la materia.

## 1. Librerías y configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import beta, gamma, norm
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, FloatSlider, IntSlider, HBox, VBox, Output
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings("ignore")

# ── Estilo UNS ────────────────────────────────────────────────────────────────
UNS_BLUE   = "#1E3C78"
UNS_ORANGE = "#B43C14"
UNS_GREEN  = "#2E7D32"
UNS_GRAY   = "#757575"

plt.rcParams.update({
    "figure.dpi"       : 120,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.size"        : 11,
    "axes.titlesize"   : 12,
    "axes.labelsize"   : 11,
})

print("✓ Librerías cargadas correctamente.")
print("  Versión de ipywidgets:", widgets.__version__)

---
## 2. Ejemplo 1 — Bernoulli–Beta

### Modelo
Sea $X_1, \ldots, X_n \sim \text{Bernoulli}(\theta)$, con $\theta \in (0,1)$ desconocido.

**Prior conjugada:** $\theta \sim \text{Beta}(\alpha, \beta)$, con media a priori $\mu_0 = \frac{\alpha}{\alpha+\beta}$.

**Posterior:** Dada la muestra con $k = n\bar{x}$ éxitos:
$$\theta \mid \mathbf{x} \sim \text{Beta}(\alpha + k,\; \beta + n - k)$$

**Estimadores:**

| Estimador | Expresión | Bajo pérdida... |
|-----------|-----------|-----------------|
| Bayesiano (media posterior) | $\hat{\theta}_B = \frac{\alpha + k}{\alpha + \beta + n}$ | Cuadrática |
| Bayesiano (mediana posterior) | mediana de Beta$(\alpha+k, \beta+n-k)$ | Error absoluto |
| MAP (Máximo a Posteriori) | $\frac{\alpha+k-1}{\alpha+\beta+n-2}$ | — |
| Clásico (MV) | $\hat{\theta}_{MV} = \bar{x} = k/n$ | — |

> **Clave:** $\hat{\theta}_B$ es una **media ponderada** entre la media a priori y $\bar{x}$.  
> A medida que $n \to \infty$, $\hat{\theta}_B \to \bar{x}$ (los datos dominan la prior).

In [ ]:
# ── Datos simulados (fijos para reproducibilidad) ─────────────────────────────
# La semilla y la proporción verdadera se pueden cambiar aquí
SEED_BB   = 42
THETA_VER = 0.35   # valor verdadero de θ (desconocido para el "analista")

np.random.seed(SEED_BB)
datos_bb_full = np.random.binomial(1, THETA_VER, size=500)  # banco de 500 observaciones

print(f"Banco de datos generado: 500 observaciones Bernoulli(θ={THETA_VER})")
print(f"Proporción muestral global (n=500): {datos_bb_full.mean():.4f}")

In [ ]:
def plot_bernoulli_beta(alpha_prior, beta_prior, n):
    """
    Visualiza prior, verosimilitud normalizada y posterior para el modelo Bernoulli-Beta.
    Compara estimadores bayesianos con el estimador clásico MV.
    """
    # ── Datos ──────────────────────────────────────────────────────────────────
    datos = datos_bb_full[:n]
    k     = datos.sum()          # número de éxitos
    xbar  = k / n                # media muestral = estimador MV

    # ── Parámetros a posteriori ────────────────────────────────────────────────
    alpha_post = alpha_prior + k
    beta_post  = beta_prior  + n - k

    # ── Estimadores bayesianos ─────────────────────────────────────────────────
    est_media   = alpha_post / (alpha_post + beta_post)          # pérdida cuadrática
    est_mediana = stats.beta.ppf(0.5, alpha_post, beta_post)     # pérdida absoluta
    est_map     = (alpha_post - 1) / (alpha_post + beta_post - 2) if (alpha_post + beta_post) > 2 else np.nan

    # ── Grilla ────────────────────────────────────────────────────────────────
    theta = np.linspace(0.001, 0.999, 500)

    prior_pdf  = stats.beta.pdf(theta, alpha_prior, beta_prior)
    post_pdf   = stats.beta.pdf(theta, alpha_post,  beta_post)

    # Verosimilitud (log para estabilidad, luego normalizar)
    log_vero = k * np.log(theta) + (n - k) * np.log(1 - theta)
    log_vero -= log_vero.max()
    vero_norm = np.exp(log_vero)
    vero_norm = vero_norm / np.trapezoid(vero_norm, theta)   # normalizar para graficar

    # ── Figura ────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(15, 9))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

    # Panel 1: Prior
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(theta, prior_pdf, color=UNS_GRAY, lw=2.5)
    ax1.axvline(alpha_prior / (alpha_prior + beta_prior), color=UNS_GRAY,
                ls="--", lw=1.5, label=f"Media a priori = {alpha_prior/(alpha_prior+beta_prior):.3f}")
    ax1.axvline(THETA_VER, color="black", ls=":", lw=1.2, alpha=0.5, label=f"θ verdadero = {THETA_VER}")
    ax1.set_title(f"Prior: Beta({alpha_prior:.1f}, {beta_prior:.1f})")
    ax1.set_xlabel("θ"); ax1.set_ylabel("Densidad")
    ax1.legend(fontsize=9)

    # Panel 2: Verosimilitud
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(theta, vero_norm, color=UNS_ORANGE, lw=2.5)
    ax2.axvline(xbar, color=UNS_ORANGE, ls="--", lw=1.5, label=f"MV = x̄ = {xbar:.3f}")
    ax2.axvline(THETA_VER, color="black", ls=":", lw=1.2, alpha=0.5, label=f"θ verdadero = {THETA_VER}")
    ax2.set_title(f"Verosimilitud normalizada  (n={n}, k={k})")
    ax2.set_xlabel("θ"); ax2.set_ylabel("Densidad")
    ax2.legend(fontsize=9)

    # Panel 3: Posterior + comparación
    ax3 = fig.add_subplot(gs[1, :])
    ax3.plot(theta, prior_pdf, color=UNS_GRAY,   lw=1.5, ls="--", alpha=0.7, label=f"Prior Beta({alpha_prior:.1f},{beta_prior:.1f})")
    ax3.plot(theta, vero_norm, color=UNS_ORANGE, lw=1.5, ls="--", alpha=0.7, label=f"Verosimilitud (normalizada)")
    ax3.plot(theta, post_pdf,  color=UNS_BLUE,   lw=2.5,           label=f"Posterior Beta({alpha_post:.1f},{beta_post:.1f})")

    ax3.axvline(THETA_VER,   color="black",      ls=":",  lw=1.5, alpha=0.6, label=f"θ verdadero = {THETA_VER}")
    ax3.axvline(xbar,        color=UNS_ORANGE,   ls="--", lw=2,   label=f"MV (clásico) = {xbar:.4f}")
    ax3.axvline(est_media,   color=UNS_BLUE,     ls="-",  lw=2,   label=f"Bayes media (cuadrática) = {est_media:.4f}")
    ax3.axvline(est_mediana, color=UNS_GREEN,    ls="-.", lw=1.8, label=f"Bayes mediana (abs.) = {est_mediana:.4f}")
    if not np.isnan(est_map):
        ax3.axvline(est_map, color="purple", ls=":", lw=1.8, label=f"MAP = {est_map:.4f}")

    ax3.set_title("Prior · Verosimilitud · Posterior  —  comparación de estimadores", fontsize=13)
    ax3.set_xlabel("θ"); ax3.set_ylabel("Densidad")
    ax3.legend(fontsize=9, ncol=2)

    # Caja de texto con resumen
    sesgo_b = est_media - THETA_VER
    sesgo_mv = xbar - THETA_VER
    info = (f"Prior: media={alpha_prior/(alpha_prior+beta_prior):.3f}  "
            f"(α+β={alpha_prior+beta_prior:.0f} 'obs. virtuales')\n"
            f"MV:    {xbar:.4f}   sesgo={sesgo_mv:+.4f}\n"
            f"Bayes: {est_media:.4f}   sesgo={sesgo_b:+.4f}\n"
            f"Peso prior en Bayes: {(alpha_prior+beta_prior)/(alpha_prior+beta_prior+n)*100:.1f}%")
    ax3.text(0.02, 0.97, info, transform=ax3.transAxes, fontsize=9,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.suptitle(f"Bernoulli–Beta  |  n = {n}  |  k = {k} éxitos  |  x̄ = {xbar:.3f}", fontsize=14)
    plt.savefig(f"/tmp/bb_n{n}.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()

print("✓ Función plot_bernoulli_beta definida.")

In [ ]:
# ── Widget interactivo — Bernoulli-Beta ───────────────────────────────────────
w_alpha = FloatSlider(value=2.0,  min=0.5, max=20.0, step=0.5,
                      description="α (prior)", style={"description_width":"100px"},
                      layout=widgets.Layout(width="420px"))
w_beta  = FloatSlider(value=2.0,  min=0.5, max=20.0, step=0.5,
                      description="β (prior)", style={"description_width":"100px"},
                      layout=widgets.Layout(width="420px"))
w_n_bb  = IntSlider(value=20,    min=1,   max=500,  step=5,
                    description="n (muestra)", style={"description_width":"100px"},
                    layout=widgets.Layout(width="420px"))

out_bb = Output()

def update_bb(alpha_prior, beta_prior, n):
    with out_bb:
        clear_output(wait=True)
        plot_bernoulli_beta(alpha_prior, beta_prior, n)

ui_bb = VBox([
    widgets.HTML("<b>Ajustá los hiperparámetros de la prior Beta(α, β) y el tamaño muestral:</b>"),
    widgets.HTML("α+β = 'observaciones virtuales' que aporta la prior. α/(α+β) = media a priori."),
    HBox([w_alpha, w_beta, w_n_bb]),
])
widgets.interactive_output(update_bb, {"alpha_prior": w_alpha, "beta_prior": w_beta, "n": w_n_bb})
display(ui_bb, out_bb)
update_bb(w_alpha.value, w_beta.value, w_n_bb.value)

### ¿Qué observar?
- Con **α = β = 1** (prior uniforme): la posterior coincide con la verosimilitud; el estimador bayesiano y el MV son casi idénticos.
- Con **α+β grande**: la prior es muy informativa y "arrastra" el estimador bayesiano hacia la media a priori.
- Al **aumentar n**: los datos dominan la prior; todos los estimadores convergen a θ verdadero.
- El **peso de la prior** en el estimador bayesiano es exactamente $\frac{\alpha+\beta}{\alpha+\beta+n}$.

---
## 3. Ejemplo 2 — Exponencial–Gamma

### Modelo
Sea $X_1, \ldots, X_n \sim \mathcal{E}(\lambda)$, con $\lambda > 0$ (tasa) desconocida.  
Media poblacional: $\mu = 1/\lambda$.

**Prior conjugada:** $\lambda \sim \text{Gamma}(\alpha, \beta)$, con  
$E(\lambda) = \alpha/\beta$ y $V(\lambda) = \alpha/\beta^2$.

**Posterior:**
$$\lambda \mid \mathbf{x} \sim \text{Gamma}\!\left(\alpha + n,\; \beta + \sum_{i=1}^n x_i\right)$$

**Estimadores:**

| Estimador | Expresión |
|-----------|-----------|
| Bayesiano (media posterior) | $\hat{\lambda}_B = \frac{\alpha + n}{\beta + \sum x_i}$ |
| MAP | $\frac{\alpha + n - 1}{\beta + \sum x_i}$ |
| Clásico (MV) | $\hat{\lambda}_{MV} = n / \sum x_i = 1/\bar{x}$ |

> **Ejemplo de clase:** lámparas con $\mu \approx 5000$ h → $E(\lambda) = 1/5000$,  
> elegimos $\alpha=4$, $\beta=20000$, con lo que $V(\lambda)=10^{-8}$.

In [ ]:
# ── Datos simulados ───────────────────────────────────────────────────────────
SEED_EG    = 7
LAMBDA_VER = 1/5000   # valor verdadero

np.random.seed(SEED_EG)
datos_eg_full = np.random.exponential(scale=1/LAMBDA_VER, size=200)

print(f"Datos generados: 200 obs Exponencial(λ={LAMBDA_VER:.6f}  →  media={1/LAMBDA_VER:.0f} h)")
print(f"Media muestral global (n=200): {datos_eg_full.mean():.1f} h")

In [ ]:
def plot_exponencial_gamma(alpha_prior, beta_prior, n):
    """
    Visualiza prior, verosimilitud normalizada y posterior para Exponencial-Gamma.
    """
    datos  = datos_eg_full[:n]
    sum_x  = datos.sum()
    xbar   = datos.mean()
    lam_mv = 1 / xbar    # estimador MV de λ

    # Parámetros a posteriori
    alpha_post = alpha_prior + n
    beta_post  = beta_prior  + sum_x

    # Estimadores bayesianos
    est_media = alpha_post / beta_post                          # pérdida cuadrática
    est_map   = (alpha_post - 1) / beta_post if alpha_post > 1 else np.nan

    # Media estimada de X = 1/λ
    mu_bayes = 1 / est_media
    mu_mv    = xbar

    # Grilla en λ (cerca del valor verdadero)
    lam_max = max(lam_mv, est_media) * 5
    lam_min = max(1e-8, min(lam_mv, est_media) / 5)
    lam_grid = np.linspace(lam_min, lam_max, 600)

    prior_pdf = stats.gamma.pdf(lam_grid, a=alpha_prior, scale=1/beta_prior)
    post_pdf  = stats.gamma.pdf(lam_grid, a=alpha_post,  scale=1/beta_post)

    # Verosimilitud log-normalizada
    log_vero = n * np.log(lam_grid) - lam_grid * sum_x
    log_vero -= log_vero.max()
    vero_norm = np.exp(log_vero)
    vero_norm /= np.trapezoid(vero_norm, lam_grid)

    # ── Figura ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Panel izquierdo: en escala de λ
    ax = axes[0]
    ax.plot(lam_grid, prior_pdf, color=UNS_GRAY,   lw=1.8, ls="--", alpha=0.8,
            label=f"Prior Gamma({alpha_prior:.1f},{beta_prior:.0f})")
    ax.plot(lam_grid, vero_norm, color=UNS_ORANGE, lw=1.8, ls="--", alpha=0.8,
            label="Verosimilitud (normalizada)")
    ax.plot(lam_grid, post_pdf,  color=UNS_BLUE,   lw=2.5,
            label=f"Posterior Gamma({alpha_post:.1f},{beta_post:.0f})")
    ax.axvline(LAMBDA_VER, color="black",    ls=":",  lw=1.5, alpha=0.6,
               label=f"λ verdadero = {LAMBDA_VER:.2e}")
    ax.axvline(lam_mv,     color=UNS_ORANGE, ls="--", lw=2,
               label=f"MV: λ̂ = {lam_mv:.2e}")
    ax.axvline(est_media,  color=UNS_BLUE,   ls="-",  lw=2,
               label=f"Bayes (media): λ̂ = {est_media:.2e}")
    if not np.isnan(est_map):
        ax.axvline(est_map, color="purple", ls=":", lw=1.8,
                   label=f"MAP: λ̂ = {est_map:.2e}")
    ax.set_title(f"Distribuciones sobre λ  (n={n})")
    ax.set_xlabel("λ (tasa)"); ax.set_ylabel("Densidad")
    ax.legend(fontsize=8.5)

    # Panel derecho: media estimada de X = 1/λ (más intuitivo)
    ax2 = axes[1]
    mu_grid = 1 / lam_grid[::-1]   # invertir
    ax2.axvline(1/LAMBDA_VER, color="black",    ls=":",  lw=1.5, alpha=0.6,
                label=f"μ verdadero = {1/LAMBDA_VER:.0f} h")
    ax2.axvline(mu_mv,        color=UNS_ORANGE, ls="--", lw=2,
                label=f"MV: μ̂ = 1/λ̂ = {mu_mv:.0f} h")
    ax2.axvline(mu_bayes,     color=UNS_BLUE,   ls="-",  lw=2,
                label=f"Bayes: μ̂ = 1/E(λ|x) = {mu_bayes:.0f} h")

    # Histograma de los datos para contexto
    ax2.hist(datos, bins=30, density=True, color=UNS_BLUE, alpha=0.2,
             edgecolor="white", label=f"Datos (n={n})")
    # Curva exponencial con λ_Bayes
    x_plot = np.linspace(0, datos.max()*1.2, 300)
    ax2.plot(x_plot, est_media * np.exp(-est_media * x_plot), color=UNS_BLUE, lw=2,
             label=f"Exp(λ_Bayes)")
    ax2.plot(x_plot, lam_mv * np.exp(-lam_mv * x_plot), color=UNS_ORANGE, lw=2, ls="--",
             label=f"Exp(λ_MV)")
    ax2.set_title(f"Datos y modelos ajustados  (escala μ=1/λ)")
    ax2.set_xlabel("Tiempo de vida (horas)"); ax2.set_ylabel("Densidad")
    ax2.set_xlim(0, min(datos.max()*1.2, 30000))
    ax2.legend(fontsize=8.5)

    # Resumen
    mu_prior = alpha_prior / beta_prior
    info = (f"Prior: E(λ)={alpha_prior/beta_prior:.2e}  →  E(μ)≈{beta_prior/alpha_prior:.0f} h\n"
            f"MV:    μ̂={mu_mv:.0f} h\n"
            f"Bayes: μ̂={mu_bayes:.0f} h  (error={mu_bayes-1/LAMBDA_VER:+.0f} h)\n"
            f"Peso prior: {beta_prior/(beta_prior+sum_x)*100:.1f}%")
    axes[0].text(0.98, 0.97, info, transform=axes[0].transAxes, fontsize=9,
                 ha="right", va="top", bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

    plt.suptitle(f"Exponencial–Gamma  |  n={n}  |  Σxᵢ={sum_x:.0f}  |  x̄={xbar:.0f} h", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"/tmp/eg_n{n}.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()

print("✓ Función plot_exponencial_gamma definida.")

In [ ]:
# ── Widget interactivo — Exponencial-Gamma ────────────────────────────────────
# Prior del ejemplo de clase: α=4, β=20000  (E(λ)=1/5000, V(λ)=10^-8)
w_alpha_eg = FloatSlider(value=4.0,   min=1.0, max=50.0,    step=1.0,
                         description="α (prior)", style={"description_width":"100px"},
                         layout=widgets.Layout(width="420px"))
w_beta_eg  = FloatSlider(value=20000, min=1000, max=100000, step=1000,
                         description="β (prior)", style={"description_width":"100px"},
                         layout=widgets.Layout(width="420px"))
w_n_eg     = IntSlider(value=10,  min=1, max=200, step=5,
                       description="n (muestra)", style={"description_width":"100px"},
                       layout=widgets.Layout(width="420px"))

out_eg = Output()

def update_eg(alpha_prior, beta_prior, n):
    with out_eg:
        clear_output(wait=True)
        plot_exponencial_gamma(alpha_prior, beta_prior, n)

ui_eg = VBox([
    widgets.HTML("<b>Prior Gamma(α, β): E(λ)=α/β — ejemplo de clase: α=4, β=20000</b>"),
    widgets.HTML("β ≈ suma de tiempos 'virtuales' aportados por la prior (en horas)."),
    HBox([w_alpha_eg, w_beta_eg, w_n_eg]),
])
widgets.interactive_output(update_eg, {"alpha_prior": w_alpha_eg, "beta_prior": w_beta_eg, "n": w_n_eg})
display(ui_eg, out_eg)
update_eg(w_alpha_eg.value, w_beta_eg.value, w_n_eg.value)

### ¿Qué observar?
- Con **α=1, β=1** (prior casi no informativa): el estimador bayesiano converge rápido al MV.
- Con **α=4, β=20000** (prior del ejemplo de clase): con pocas observaciones el estimador bayesiano queda "anclado" cerca de 5000 h.
- Al **aumentar n**: la suma $\sum x_i$ domina sobre $\beta$ en el denominador de la posterior, y el estimador bayesiano se acerca al MV.
- El **peso de la prior** en términos de "horas virtuales" es $\beta/(\beta + \sum x_i)$.

---
## 4. Ejemplo 3 — Normal–Normal

### Modelo
Sea $X_1, \ldots, X_n \sim \mathcal{N}(\theta, \sigma^2)$, con $\sigma^2$ **conocida** y $\theta$ desconocido.

**Prior conjugada:** $\theta \sim \mathcal{N}(\mu_0, \nu^2)$.

**Posterior:**
$$\theta \mid \mathbf{x} \sim \mathcal{N}(\mu_1, \nu_1^2)$$
con
$$\mu_1 = \frac{\sigma^2}{\sigma^2 + n\nu^2}\,\mu_0 + \frac{n\nu^2}{\sigma^2 + n\nu^2}\,\bar{x}, \qquad
\nu_1^2 = \frac{\sigma^2\nu^2}{\sigma^2 + n\nu^2}.$$

**Estimadores:**

| Estimador | Expresión |
|-----------|-----------|
| Bayesiano (= media = MAP, por simetría) | $\hat{\theta}_B = \mu_1$ |
| Clásico (MV) | $\hat{\theta}_{MV} = \bar{x}$ |

> La media posterior $\mu_1$ es una **combinación convexa** de $\mu_0$ y $\bar{x}$.  
> El peso de $\bar{x}$ aumenta con $n$ y disminuye con $\nu^2$ (prior más difusa → más peso a los datos).

In [ ]:
# ── Datos simulados ───────────────────────────────────────────────────────────
SEED_NN  = 13
THETA_VER_NN = 72.0   # valor verdadero de θ (p.ej. presión arterial sistólica media)
SIGMA2   = 100.0      # varianza conocida (σ=10 mmHg)

np.random.seed(SEED_NN)
datos_nn_full = np.random.normal(THETA_VER_NN, np.sqrt(SIGMA2), size=300)

print(f"Datos generados: 300 obs N(θ={THETA_VER_NN}, σ²={SIGMA2})")
print(f"Media muestral global (n=300): {datos_nn_full.mean():.4f}")

In [ ]:
def plot_normal_normal(mu0, nu2, n, sigma2=SIGMA2):
    """
    Visualiza prior, verosimilitud y posterior para el modelo Normal-Normal.
    """
    datos = datos_nn_full[:n]
    xbar  = datos.mean()

    # Parámetros a posteriori
    w_data  = (n * nu2) / (sigma2 + n * nu2)    # peso de los datos
    w_prior = sigma2   / (sigma2 + n * nu2)      # peso de la prior
    mu1     = w_prior * mu0 + w_data * xbar
    nu1_2   = (sigma2 * nu2) / (sigma2 + n * nu2)
    nu1     = np.sqrt(nu1_2)

    # IC 95% posterior
    ic_lo = mu1 - 1.96 * nu1
    ic_hi = mu1 + 1.96 * nu1

    # Grilla
    rango  = max(4 * np.sqrt(nu2), 4 * np.sqrt(sigma2/n), abs(mu0 - THETA_VER_NN) + 20)
    center = (mu0 + xbar + THETA_VER_NN) / 3
    theta_grid = np.linspace(center - rango, center + rango, 600)

    prior_pdf = stats.norm.pdf(theta_grid, mu0, np.sqrt(nu2))
    post_pdf  = stats.norm.pdf(theta_grid, mu1, nu1)
    vero_pdf  = stats.norm.pdf(theta_grid, xbar, np.sqrt(sigma2/n))   # distribución de x̄

    # ── Figura ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Panel izquierdo: prior + verosimilitud + posterior
    ax = axes[0]
    ax.plot(theta_grid, prior_pdf, color=UNS_GRAY,   lw=2, ls="--", alpha=0.8,
            label=f"Prior N({mu0:.1f}, {nu2:.0f})  [σ_prior={np.sqrt(nu2):.1f}]")
    ax.plot(theta_grid, vero_pdf,  color=UNS_ORANGE, lw=2, ls="--", alpha=0.8,
            label=f"Verosim. de x̄  N(x̄, σ²/n)  [σ={np.sqrt(sigma2/n):.2f}]")
    ax.plot(theta_grid, post_pdf,  color=UNS_BLUE,   lw=2.5,
            label=f"Posterior N({mu1:.3f}, {nu1_2:.3f})")

    ax.axvline(THETA_VER_NN, color="black",    ls=":",  lw=1.5, alpha=0.6,
               label=f"θ verdadero = {THETA_VER_NN}")
    ax.axvline(xbar,         color=UNS_ORANGE, ls="--", lw=2,
               label=f"MV: x̄ = {xbar:.3f}")
    ax.axvline(mu1,          color=UNS_BLUE,   ls="-",  lw=2,
               label=f"Bayes: μ₁ = {mu1:.3f}")
    ax.axvline(mu0,          color=UNS_GRAY,   ls=":",  lw=1.5,
               label=f"Media prior: μ₀ = {mu0:.1f}")

    # IC posterior sombreado
    ic_mask = (theta_grid >= ic_lo) & (theta_grid <= ic_hi)
    ax.fill_between(theta_grid[ic_mask], post_pdf[ic_mask], alpha=0.15, color=UNS_BLUE,
                    label=f"IC 95% post: [{ic_lo:.2f}, {ic_hi:.2f}]")

    ax.set_title(f"Prior · Verosimilitud · Posterior  (n={n})", fontsize=12)
    ax.set_xlabel("θ"); ax.set_ylabel("Densidad")
    ax.legend(fontsize=8.5)

    # Panel derecho: evolución de μ₁ con n
    ns = np.arange(1, 301)
    w_data_n  = (ns * nu2) / (sigma2 + ns * nu2)
    mu1_n     = (1 - w_data_n) * mu0 + w_data_n * xbar
    nu1_n     = np.sqrt((sigma2 * nu2) / (sigma2 + ns * nu2))

    ax2 = axes[1]
    ax2.plot(ns, mu1_n, color=UNS_BLUE, lw=2, label="Estimador bayesiano μ₁(n)")
    ax2.fill_between(ns, mu1_n - 1.96*nu1_n, mu1_n + 1.96*nu1_n,
                     alpha=0.15, color=UNS_BLUE, label="IC 95% posterior")
    ax2.axhline(xbar,          color=UNS_ORANGE, ls="--", lw=1.8, label=f"MV global (x̄={xbar:.2f})")
    ax2.axhline(THETA_VER_NN,  color="black",    ls=":",  lw=1.5, alpha=0.6, label=f"θ verdadero={THETA_VER_NN}")
    ax2.axhline(mu0,           color=UNS_GRAY,   ls=":",  lw=1.5, label=f"Media prior μ₀={mu0:.1f}")
    ax2.axvline(n, color="red", ls="--", lw=1.2, alpha=0.7, label=f"n actual = {n}")
    ax2.set_title("Convergencia del estimador bayesiano con n")
    ax2.set_xlabel("n (tamaño muestral)"); ax2.set_ylabel("θ estimado")
    ax2.legend(fontsize=8.5)

    # Resumen numérico
    info = (f"Peso prior: {w_prior*100:.1f}%   Peso datos: {w_data*100:.1f}%\n"
            f"MV (x̄):   {xbar:.4f}  (error={xbar-THETA_VER_NN:+.4f})\n"
            f"Bayes (μ₁): {mu1:.4f}  (error={mu1-THETA_VER_NN:+.4f})\n"
            f"σ posterior: {nu1:.4f}  vs  σ prior: {np.sqrt(nu2):.4f}")
    axes[0].text(0.02, 0.97, info, transform=axes[0].transAxes, fontsize=9,
                 va="top", bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

    plt.suptitle(f"Normal–Normal  |  n={n}  |  σ²={sigma2}  |  x̄={xbar:.3f}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"/tmp/nn_n{n}.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()

print("✓ Función plot_normal_normal definida.")

In [ ]:
# ── Widget interactivo — Normal-Normal ────────────────────────────────────────
w_mu0  = FloatSlider(value=70.0,  min=50.0, max=90.0, step=1.0,
                     description="μ₀ (media prior)", style={"description_width":"130px"},
                     layout=widgets.Layout(width="440px"))
w_nu2  = FloatSlider(value=100.0, min=1.0,  max=2000.0, step=10.0,
                     description="ν² (var. prior)", style={"description_width":"130px"},
                     layout=widgets.Layout(width="440px"))
w_n_nn = IntSlider(value=10,  min=1, max=300, step=5,
                   description="n (muestra)", style={"description_width":"130px"},
                   layout=widgets.Layout(width="440px"))

out_nn = Output()

def update_nn(mu0, nu2, n):
    with out_nn:
        clear_output(wait=True)
        plot_normal_normal(mu0, nu2, n)

ui_nn = VBox([
    widgets.HTML(f"<b>Modelo: X ~ N(θ, σ²={SIGMA2})  |  Prior: θ ~ N(μ₀, ν²)</b>"),
    widgets.HTML("ν² grande = prior difusa (poco informativa). ν² pequeño = prior concentrada (muy informativa)."),
    HBox([w_mu0, w_nu2, w_n_nn]),
])
widgets.interactive_output(update_nn, {"mu0": w_mu0, "nu2": w_nu2, "n": w_n_nn})
display(ui_nn, out_nn)
update_nn(w_mu0.value, w_nu2.value, w_n_nn.value)

### ¿Qué observar?
- Con **ν² → ∞** (prior muy difusa): el estimador bayesiano converge al MV desde el primer dato.
- Con **ν² pequeño** (prior muy concentrada): se necesitan muchos datos para que la posterior se aleje de μ₀.
- Con **μ₀ lejos del verdadero θ**: una prior sesgada requiere muchos datos para "corregirse".
- El panel derecho muestra la **convergencia de μ₁ hacia x̄** a medida que n crece — con ν² grande, la convergencia es instantánea; con ν² pequeño, es lenta.

---
## 5. Síntesis — Comparación de los tres modelos

### Estructura común
En los tres ejemplos, el estimador bayesiano (bajo pérdida cuadrática) es siempre 
una **combinación convexa** entre la media a priori y el estimador clásico:

$$\hat{\theta}_B = w_{\text{prior}} \cdot \mu_0 + w_{\text{datos}} \cdot \hat{\theta}_{MV}$$

| Modelo | $w_{\text{prior}}$ | $w_{\text{datos}}$ | Prior conjugada |
|--------|---------------------|---------------------|-----------------|
| Bernoulli–Beta | $\frac{\alpha+\beta}{\alpha+\beta+n}$ | $\frac{n}{\alpha+\beta+n}$ | Beta(α, β) |
| Exponencial–Gamma | $\frac{\beta}{\beta+\sum x_i}$ | $\frac{\sum x_i}{\beta+\sum x_i}$ | Gamma(α, β) |
| Normal–Normal | $\frac{\sigma^2}{\sigma^2+n\nu^2}$ | $\frac{n\nu^2}{\sigma^2+n\nu^2}$ | N(μ₀, ν²) |

### Conclusiones generales
- **Con n grande**, todos los estimadores bayesianos convergen al estimador clásico (MV).
- **Con prior informativa correcta** (μ₀ cerca del verdadero θ) y **n pequeño**, el estimador bayesiano es más preciso que el MV.
- **Con prior mal especificada** y **n pequeño**, el estimador bayesiano puede ser peor que el MV.
- La elección de la prior es más crítica cuando el tamaño muestral es pequeño.

> 📖 Ver Sección 10.3–10.5 de los apuntes — Estimadores Bayesianos y función de pérdida.

In [ ]:
# ── Figura de síntesis: convergencia del peso de la prior con n ───────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

ns = np.arange(1, 201)

# Bernoulli-Beta (α=β=2)
alpha_b, beta_b = 2, 2
w_prior_bb = (alpha_b + beta_b) / (alpha_b + beta_b + ns)
axes[0].plot(ns, w_prior_bb * 100, color=UNS_BLUE, lw=2.5, label="α=β=2")
alpha_b2, beta_b2 = 10, 10
w_prior_bb2 = (alpha_b2 + beta_b2) / (alpha_b2 + beta_b2 + ns)
axes[0].plot(ns, w_prior_bb2 * 100, color=UNS_ORANGE, lw=2, ls="--", label="α=β=10")
axes[0].set_title("Bernoulli–Beta
Peso de la prior (%)")
axes[0].set_xlabel("n"); axes[0].set_ylabel("Peso prior (%)")
axes[0].legend(); axes[0].set_ylim(0, 100)

# Exponencial-Gamma (β=20000, distintos α)
sum_x_n = (1/LAMBDA_VER) * ns   # Σxᵢ ≈ n × E(X)
beta_eg1 = 20000
w_prior_eg1 = beta_eg1 / (beta_eg1 + sum_x_n)
axes[1].plot(ns, w_prior_eg1 * 100, color=UNS_BLUE, lw=2.5, label="β=20000 (clase)")
beta_eg2 = 1000
w_prior_eg2 = beta_eg2 / (beta_eg2 + sum_x_n)
axes[1].plot(ns, w_prior_eg2 * 100, color=UNS_ORANGE, lw=2, ls="--", label="β=1000")
axes[1].set_title("Exponencial–Gamma
Peso de la prior (%)")
axes[1].set_xlabel("n"); axes[1].legend()

# Normal-Normal (σ²=100, distintos ν²)
sigma2_nn = 100
for nu2_val, col, lab in [(100, UNS_BLUE, "ν²=100"), (1000, UNS_ORANGE, "ν²=1000"), (10, UNS_GREEN, "ν²=10")]:
    w_p = sigma2_nn / (sigma2_nn + ns * nu2_val)
    axes[2].plot(ns, w_p * 100, color=col, lw=2, label=lab)
axes[2].set_title("Normal–Normal
Peso de la prior (%)")
axes[2].set_xlabel("n"); axes[2].legend()

for ax in axes:
    ax.axhline(50, color="gray", ls=":", lw=1, alpha=0.5)
    ax.text(axes[0].get_xlim()[1]*0.98, 51, "50%", fontsize=8, color="gray", ha="right")

plt.suptitle("Convergencia del estimador bayesiano al estimador clásico\n"
             "Peso de la prior en función del tamaño muestral", fontsize=13)
plt.tight_layout()
plt.savefig("/tmp/sintesis_convergencia.png", dpi=130, bbox_inches="tight")
plt.show()
print("A medida que n crece, el peso de la prior cae a cero y el estimador bayesiano converge al MV.")

---
## Resumen de secciones

| Sección | Modelo | Prior conjugada | Posterior | Estimador bayesiano |
|---------|--------|-----------------|-----------|---------------------|
| 2 | Bernoulli(θ) | Beta(α, β) | Beta(α+k, β+n-k) | (α+k)/(α+β+n) |
| 3 | Exponencial(λ) | Gamma(α, β) | Gamma(α+n, β+Σxᵢ) | (α+n)/(β+Σxᵢ) |
| 4 | Normal(θ, σ²) | N(μ₀, ν²) | N(μ₁, ν₁²) | combinación convexa |

**Para explorar:** modificá los sliders de cada sección y prestá atención a:
1. Cómo cambia la forma de la posterior cuando la prior es más o menos informativa.
2. Cuántos datos se necesitan para que el estimador bayesiano "olvide" la prior.
3. Qué pasa cuando la prior está mal especificada (media a priori alejada del verdadero θ).
